### 1. Install Unsloth and Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

### 2. Load Base Model and Tokenizer
Swap `model_name` to train other models.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Lower to save memory
dtype = None # None for auto detection.
load_in_4bit = True

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    # Change model here
    model_name = "unsloth/gpt-oss-20b-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Prepare Dataset

In [ ]:
# Load the GAD dataset
import json

with open('gad_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')

In [ ]:
from sklearn.model_selection import train_test_split

# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(len(conv_train), len(_conv_test), len(score_train), len(_score_test))

In [ ]:
from datasets import Dataset
import json

prompt = """
You are for a GAD-7 survey chatbot.
Your job is to extract answers for GAD-7 based on a given transcript.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a widely used self-administered diagnostic tool designed to screen for and assess the severity of generalized anxiety disorder (GAD).
GAD-7 question order:
q1: Over the last two weeks, how often have you been bothered by feeling nervous, anxious, or on edge?
q2: Over the last two weeks, how often have you been bothered by not being able to stop or control worrying?
q3: Over the last two weeks, how often have you been bothered by worrying too much about different things?
q4: Over the last two weeks, how often have you been bothered by trouble relaxing?
q5: Over the last two weeks, how often have you been bothered by being so restless that it is hard to sit still?
q6: Over the last two weeks, how often have you been bothered by becoming easily annoyed or irritable?
q7: Over the last two weeks, how often have you been bothered by feeling afraid, as if something awful might happen?
Scale mapping:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day


Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
The final normalized answer for each question must be a single string value: "0", "1", "2", or "3".
Rules:
- For each question q1-q7, determine the score 0-3 based on the user's responses.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any q1-q7 key.
- expected_output must contain exactly q1 through q7.
- conversation must be consistent with expected_output.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0"
}
"""

EOS_TOKEN = tokenizer.eos_token

def format_dataset(conversations, scores):
    texts = []
    for conv, score in zip(conversations, scores):
        conv_str = json.dumps(conv, ensure_ascii=False)
        score_str = json.dumps(score, ensure_ascii=False)
        text = prompt + f"\n\nInput:\n{conv_str}\n\nOutput:\n{score_str}" + EOS_TOKEN
        texts.append(text)
    return texts

# Format the training split
formatted_texts = format_dataset(conv_train, score_train)

# Convert to a Hugging Face Dataset object which SFTTrainer requires
dataset = Dataset.from_dict({"text": formatted_texts})

print(f"Successfully created HuggingFace Dataset with {len(dataset)} examples.")

### 4. Train the Model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 3,
        warmup_steps = 5,
        num_train_epochs = 4,
        learning_rate = 2e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()


### 5. Export to Ollama (GGUF)

In [ ]:
# Save to 8bit Q8_0
model.save_pretrained_gguf("model_for_ollama", tokenizer, quantization_method = "q8_0")

print("Done! You can now download the .gguf file from the 'model_for_ollama' folder and run it in Ollama using a Modelfile.")